# Stroke Risk — Modelling

We compare two models suited to small, imbalanced tabular data:

- **Logistic Regression** — an interpretable clinical baseline.
- **XGBoost** — gradient-boosted trees that capture non-linear interactions.

**Discipline that keeps this honest:**
- every model is a pipeline, so preprocessing is fit on training folds only;
- imbalance is handled with **class weights** (no resampling leakage);
- we lead with **PR-AUC** and **recall** (right metrics for a rare, high-cost event);
- the decision **threshold is tuned on validation**, and the **test set is touched exactly once**, at the very end.

## Setup

In [ ]:
import warnings

import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_validate

from stroke_risk import data, evaluate, model
from stroke_risk import plotting as pl
from stroke_risk.features import split_features_target

pl.set_theme()
warnings.filterwarnings('ignore')

splits = data.split_data(data.load_raw())
X_train, y_train = split_features_target(splits.train)
X_val, y_val = split_features_target(splits.val)
X_test, y_test = split_features_target(splits.test)

models = model.build_models(y_train)
list(models)

## 1. Cross-validated comparison

5-fold stratified cross-validation on the **training set only** — a fair comparison before we ever look at validation or test.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = {}
for name, pipe in models.items():
    r = cross_validate(pipe, X_train, y_train, cv=cv, n_jobs=-1,
                       scoring=['average_precision', 'roc_auc'])
    cv_scores[name] = {
        'PR-AUC': r['test_average_precision'].mean(),
        'PR-AUC_std': r['test_average_precision'].std(),
        'ROC-AUC': r['test_roc_auc'].mean(),
    }
import pandas as pd
pd.DataFrame(cv_scores).T.round(3)

In [ ]:
names = list(cv_scores)
pr = [cv_scores[n]['PR-AUC'] for n in names]
err = [cv_scores[n]['PR-AUC_std'] for n in names]

fig, ax = plt.subplots(figsize=(7, 3.6))
bars = ax.bar(names, pr, yerr=err, color=[pl.MUTED, pl.ACCENT], capsize=5)
ax.bar_label(bars, fmt='%.3f', padding=8, color=pl.SUBTLE, fontsize=10)
baseline = y_train.mean()
ax.axhline(baseline, color=pl.SUBTLE, ls='--', lw=1)
ax.text(1.4, baseline, f'random = {baseline:.2f}', color=pl.SUBTLE, fontsize=9, va='bottom', ha='right')
ax.set_yticks([])
ax.grid(False)
pl.despine(ax, left=True)
pl.add_titles(ax, 'XGBoost edges out logistic regression',
              'Cross-validated PR-AUC (higher is better)')
plt.show()

## 2. Fit and tune the threshold on validation

We fit on train, then choose the probability cut-off that maximises F1 **on the validation set** — the test set stays sealed.

In [ ]:
val_results = {}
thresholds = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    proba_val = pipe.predict_proba(X_val)[:, 1]
    thr = evaluate.choose_threshold(y_val, proba_val)
    thresholds[name] = thr
    val_results[name] = evaluate.summarize(y_val, proba_val, thr)

evaluate.metrics_table(val_results)

## 3. Final evaluation on the held-out test set

The winner is chosen by **validation** PR-AUC. Only now do we score the test set — once.

In [ ]:
winner = max(val_results, key=lambda n: val_results[n]['PR-AUC'])
best_model = models[winner]
best_thr = thresholds[winner]

proba_test = best_model.predict_proba(X_test)[:, 1]
test_metrics = evaluate.summarize(y_test, proba_test, best_thr)
print(f'Winner: {winner}   (threshold = {best_thr:.3f})')
evaluate.metrics_table({winner: test_metrics})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
evaluate.plot_pr_curve(axes[0], y_test, proba_test, winner)
axes[0].set_title('Precision–Recall', loc='left', fontsize=12, fontweight='bold', color=pl.INK)
evaluate.plot_roc_curve(axes[1], y_test, proba_test, winner)
axes[1].set_title('ROC', loc='left', fontsize=12, fontweight='bold', color=pl.INK)
fig.text(0, 1.02, 'High ROC-AUC hides a hard precision–recall trade-off',
         fontsize=15, fontweight='bold', color=pl.INK)
plt.tight_layout()
plt.show()

In [ ]:
y_pred_test = (proba_test >= best_thr).astype(int)

fig, ax = plt.subplots(figsize=(5, 4.5))
evaluate.plot_confusion_matrix(ax, y_test, y_pred_test)
pl.add_titles(ax, f'{winner}: catching strokes at the chosen threshold',
              'Test-set confusion matrix')
plt.show()

## 4. Why the model decides (SHAP)

SHAP values show how each feature pushes an individual prediction toward or away from stroke.

In [ ]:
import shap

xgb = models['XGBoost']
pre = xgb.named_steps['preprocess']
clf = xgb.named_steps['clf']
X_train_t = pre.transform(X_train)
feature_names = pre.get_feature_names_out()

explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_train_t)
shap.summary_plot(shap_values, X_train_t, feature_names=feature_names, show=True)

## Conclusions

_Read from the results above:_

- **Stroke prediction on this dataset is genuinely hard.** ROC-AUC ~0.8 looks strong, but PR-AUC is low — exactly why we lead with PR-AUC on a rare event.
- **The threshold is a policy choice.** A lower threshold catches more strokes (higher recall) at the cost of more false alarms (lower precision); a clinician would set this, not the data scientist.
- **Age, glucose and BMI dominate** the SHAP explanation, matching the EDA.
- **Next steps:** hyperparameter tuning, calibrated probabilities, and the candidate features flagged in the EDA (glucose/BMI bands, `bmi_missing`).